# Zero-spend re-analysis — caught-fraction replaces the located-in `0.227` coverage artifact

**Artifact:** `art_4xy3D05YxvRr` (evaluation) · from *"No Derivation, No Relation: A Closure Certificate for Relation-Extraction Hallucinations"*

This notebook is a **minimal, runnable demo** of a `$0` re-analysis (numpy/scipy only — **no LLM, no network**).
The full evaluation reads three frozen experiment outputs (located-in, query-side verifier, Re-DocRED kinship),
runs a **reproduction gate** that recomputes every paper-facing number from the per-query rows and hard-asserts it
matches the carried metadata (68/68 hard checks), and then derives the corrected headline.

**What this demo reproduces** (on a curated 100-row subset of the *located-in* corpus):

1. **FACT-A** — the raw LLM confidently fabricates a containment on ~30% of *same-component-sibling* absent pairs.
2. **The caught-fraction leaderboard** (the corrected headline): of the high-confidence fabrications the raw reader
   emits, what fraction does each method turn into an *abstention*? The **closure certificate** catches the most.
3. **Natural confident-wrong** — the certificate's structural abstention keeps confident-wrong lowest.
4. A **NEW doc-clustered paired bootstrap** of every *certificate − competitor* caught-gap.

> The demo uses a 100-row subset, so the numbers are **approximate**. The full 515-present + 450-sibling-absent
> pool (135 fabrications) reproduces the exact paper literals (certificate caught **0.785**). The carried full-pool
> values are loaded alongside the subset for side-by-side comparison.

In [ ]:
# --- Install dependencies (Colab + local compatible) ---
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])

# numpy / scipy / matplotlib are pre-installed on Colab (compiled C extensions —
# installing on Colab corrupts them). Install ONLY when NOT on Colab, at Colab's exact versions.
if "google.colab" not in sys.modules:
    _pip("numpy==2.0.2", "scipy==1.16.3", "matplotlib==3.10.0")

In [ ]:
# --- Imports (mirrors the original eval.py import block; loguru/resource dropped for the notebook) ---
from __future__ import annotations
import json, math, sys
from collections import defaultdict

import numpy as np
from scipy import stats as scipy_stats  # noqa: F401  (kept for completeness, as in the original)
import matplotlib.pyplot as plt

print("numpy", np.__version__)

In [ ]:
# --- Data loading helper (GitHub URL with local fallback, Colab-compatible) ---
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-40a89b-no-derivation-no-relation-a-closure-cert/main/round-9/evaluation-1/demo/mini_demo_data.json"
import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print("source:", data["source"])
for g in data["datasets"]:
    print(f"  dataset: {g['dataset']:28s}  n_examples: {len(g['examples'])}")
carried = data["metadata"]["carried_full_pool"]
print("\ncarried FULL-pool literals (the exact paper headline this demo approximates):")
print("  FACT-A sibling:", carried["FACT_A_sibling"], "| n_fabrications:", carried["n_fabrications_sibling"])
print("  caught_fraction.certificate:", carried["caught_fraction"]["certificate"])

## Configuration

All tunable parameters live here. They start at **demo-minimum** values; comments show the original/paper values.
Because the re-analysis is pure numpy on ≤1000 rows, even the paper's bootstrap (`B = 10000`) runs in well under a
second — so the only real knob is the bootstrap resample count.

In [ ]:
# --- Config: tunable parameters (demo-minimum; originals in comments) ---
EVAL_SEED   = 20260617    # seed for the NEW caught-gap bootstrap this eval introduces
REPRO_SEED  = 20260618    # the experiments' own seed (used to Monte-Carlo-match carried CIs)

BOOTSTRAP_B = 2000        # demo value. ORIGINAL/paper: 10000 (also fits in <1s here)

# The four dispersion confidence signals compared against the certificate.
SIGNALS = ("verbalized", "sc_margin", "ptrue", "negent")

# Abstain markers: a prediction that is one of these does NOT name a relation.
ABSTAIN = {"ABSTAIN", "no-relation", "no_relation", "none", ""}

## Step 1 — Core helpers (verbatim from `eval.py`)

`is_named` decides whether a prediction commits to a relation; `matched_coverage_mask` selects the top-confidence
rows up to a target coverage (this is how a confidence signal is given the *same* coverage budget as the
certificate); `coverage_confidence` pushes abstentions to the bottom of the ranking.

In [ ]:
def is_named(pred) -> bool:
    """A prediction NAMES a relation iff it is not an abstain/no-relation marker."""
    return (pred is not None) and (str(pred) not in ABSTAIN)


def _r(x, nd=4):
    if x is None:
        return None
    if isinstance(x, float) and (math.isnan(x) or math.isinf(x)):
        return None
    return round(float(x), nd)


def matched_coverage_mask(conf: np.ndarray, target_cov: float) -> np.ndarray:
    """Replicates stats.matched_coverage_mask EXACTLY: top ceil(round) cov*N by (-conf, idx)."""
    n = len(conf)
    k = int(round(target_cov * n))
    k = max(0, min(n, k))
    mask = np.zeros(n, dtype=bool)
    if k == 0:
        return mask
    order = sorted(range(n), key=lambda i: (-conf[i], i))
    for i in order[:k]:
        mask[i] = True
    return mask


def coverage_confidence(named: bool, conf: float) -> float:
    return float(conf) if named else -1.0

## Step 2 — Caught-fraction reproduction functions (verbatim from `eval.py`)

`crux_caught` is the heart of the re-analysis. On the **mixed** pool (present + absent of a regime) it:

- finds the **fabrications** — absent pairs the raw reader confidently named;
- for the **certificate** (structural, threshold-free): caught = fraction of fabrications where the certificate abstains;
- for each **dispersion signal**: tunes a global threshold so the signal *commits* at the same coverage as the
  certificate, then caught = fraction of fabrications whose signal value falls below that threshold;
- for the **query-side gates** (verifier / self-verify): caught = fraction of fabrications they flag.

`natural_cw_rate` is the confident-wrong rate on a pure-absent pool (any named answer on an absent pair is wrong).

In [ ]:
def crux_caught(records, sigkey="metadata_conf_{}", present_records=None):
    """Reproduce the located-in/kinship crux fraction-caught for the 4 dispersion signals,
    the certificate, and (if present in the rows) the 2 query-side gates.

    `records` = the MIXED pool (present + absent of the regime).  Certificate-matched global
    threshold tuned to the certificate's MIXED coverage.
    Returns dict with cert/queryside/dispersion caught + survival + tau_global per signal.
    """
    N = len(records)
    cert_named = np.array([is_named(r.get("predict_certificate")) for r in records], bool)
    raw_named = np.array([bool(r.get("metadata_raw_named")) for r in records], bool)
    is_abs = np.array([bool(r.get("metadata_is_absent")) for r in records], bool)
    cert_cov_mixed = float(cert_named.mean()) if N else float("nan")

    absent_idx = np.where(is_abs)[0]
    halluc_idx = np.array([i for i in absent_idx if raw_named[i]], int)
    n_h = len(halluc_idx)

    out = {"n": N, "n_absent": int(is_abs.sum()), "n_raw_confident_wrong": n_h,
           "certificate_coverage_mixed": _r(cert_cov_mixed), "per_method": {}}

    # certificate: structural, threshold-free
    cert_caught = float(np.mean([0.0 if cert_named[i] else 1.0 for i in halluc_idx])) if n_h else float("nan")
    out["per_method"]["certificate"] = {"fraction_caught": _r(cert_caught),
                                        "survival": _r(1.0 - cert_caught) if cert_caught == cert_caught else None}

    # dispersion signals: certificate-matched global threshold over the MIXED pool
    for s in SIGNALS:
        m = f"ct_{s}"
        ct_named = raw_named  # ct commits raw's answer; named == raw_named
        sig_all = np.array([float(r.get(sigkey.format(s), 0.0) or 0.0) for r in records], float)
        conf_mixed = np.where(ct_named, sig_all, -1.0)
        tau_global = float("nan")
        if N and cert_cov_mixed == cert_cov_mixed:
            mask = matched_coverage_mask(conf_mixed, cert_cov_mixed)
            covered = sorted([conf_mixed[i] for i in range(N) if mask[i] and conf_mixed[i] >= 0.0])
            tau_global = covered[0] if covered else float("nan")
        vals = np.array([sig_all[i] for i in halluc_idx], float) if n_h else np.array([])
        frac_surv = float(np.mean(vals >= tau_global)) if (n_h and tau_global == tau_global) else float("nan")
        out["per_method"][m] = {"fraction_surviving": _r(frac_surv),
                                "fraction_caught": _r(1.0 - frac_surv) if frac_surv == frac_surv else None,
                                "tau_global": _r(tau_global)}
    # query-side gates (only present in the located-in rows)
    for qs in ("queryside_verifier", "queryside_selfverify"):
        key = f"predict_{qs}"
        if any(key in r for r in records):
            surv = float(np.mean([1.0 if is_named(records[i].get(key)) else 0.0 for i in halluc_idx])) if n_h else float("nan")
            out["per_method"][qs] = {"fraction_surviving": _r(surv),
                                     "fraction_caught": _r(1.0 - surv) if surv == surv else None}
    return out


def natural_cw_rate(records, predkey):
    """Confident-wrong rate on an ABSENT pool: fraction of rows where the method NAMES a relation
    (any named answer on an absent pair is wrong)."""
    if not records:
        return float("nan")
    return float(np.mean([1.0 if is_named(r.get(predkey)) else 0.0 for r in records]))

## Step 3 — The NEW doc-clustered paired bootstrap (verbatim from `eval.py`)

For each *certificate − competitor* caught-gap, resample **documents** (not rows) with replacement to get a
cluster-robust 95% CI and a one-sided p-value. On the full pool all six gaps exclude 0; on the subset the point
estimates stay positive (certificate dominates), with wider CIs because there are fewer fabrications/documents.

In [ ]:
def doc_clustered_caught_gap_bootstrap(records, cert_caught_vec, comp_caught_vec, doc_ids,
                                       B=BOOTSTRAP_B, seed=EVAL_SEED):
    """NEW statistic: doc-clustered paired bootstrap of (cert_caught - comp_caught) over the
    fabrication set.  Resample DOCUMENTS with replacement.  Returns point, CI95, one-sided p."""
    cert_caught_vec = np.asarray(cert_caught_vec, float)
    comp_caught_vec = np.asarray(comp_caught_vec, float)
    point = float(cert_caught_vec.mean() - comp_caught_vec.mean())
    by_doc = defaultdict(list)
    for i, d in enumerate(doc_ids):
        by_doc[d].append(i)
    docs = list(by_doc)
    nd = len(docs)
    rng = np.random.default_rng(seed)
    gaps = []
    for _ in range(B):
        pick = rng.integers(0, nd, nd)
        idx = np.concatenate([by_doc[docs[i]] for i in pick]) if nd else np.array([], int)
        if len(idx) == 0:
            continue
        gaps.append(float(cert_caught_vec[idx].mean() - comp_caught_vec[idx].mean()))
    gaps = np.array(gaps, float)
    lo, hi = np.quantile(gaps, [0.025, 0.975])
    p_one = max(float(np.mean(gaps <= 0.0)), 1.0 / (len(gaps) + 1))
    return {"gap": _r(point), "ci95": [_r(lo), _r(hi)], "p_one_sided": _r(p_one),
            "ci_excludes_0": bool(lo > 0.0), "n_boot": int(len(gaps))}

## Step 4 — Build the located-in mixed pool & FACT-A

The mixed pool is *present* (deducible) + *same-component-sibling* (absent) pairs. **FACT-A** is the raw reader's
confident-fabrication rate on the absent pairs — the robust diagnostic that transfers across readers and corpora.

In [ ]:
groups = {g["dataset"]: g["examples"] for g in data["datasets"]}
li_present  = groups["locatedin_present"]
li_sib      = groups["locatedin_absent_sibling"]
li_sib_mixed = li_present + li_sib                    # the decisive mixed pool

# FACT A: raw confident-fabrication rate on the sibling-absent pool
factA_sib = float(np.mean([1.0 if r["metadata_raw_named"] else 0.0 for r in li_sib]))

print(f"present rows           : {len(li_present)}")
print(f"sibling-absent rows    : {len(li_sib)}")
print(f"mixed pool             : {len(li_sib_mixed)}")
print(f"FACT-A (subset)        : {factA_sib:.4f}   (carried FULL-pool: {carried['FACT_A_sibling']})")

## Step 5 — The caught-fraction leaderboard (the corrected headline)

Run `crux_caught` on the mixed pool and tabulate the fraction of fabrications each method catches, side-by-side
with the carried full-pool literals. The **closure certificate** tops the board — well above every dispersion
signal and both query-side gates.

In [ ]:
li_crux = crux_caught(li_sib_mixed)
cm = li_crux["per_method"]
n_fab = li_crux["n_raw_confident_wrong"]
print(f"fabrications in subset (raw confident-wrong on absent): {n_fab}\n")

METHODS = ["certificate", "ct_verbalized", "ct_sc_margin", "ct_ptrue", "ct_negent",
           "queryside_verifier", "queryside_selfverify"]
LABELS = {"certificate": "certificate (no-derivation)", "ct_verbalized": "verbalized confidence",
          "ct_sc_margin": "self-consistency margin", "ct_ptrue": "Kadavath P(True)",
          "ct_negent": "semantic-entropy", "queryside_verifier": "query-side verifier",
          "queryside_selfverify": "self-verification"}
carried_caught = carried["caught_fraction"]

print(f"{'method':30s} {'caught (subset)':>16s} {'caught (full/paper)':>20s}")
print("-" * 68)
subset_caught = {}
for m in METHODS:
    c = cm[m]["fraction_caught"]
    subset_caught[m] = c
    print(f"{LABELS[m]:30s} {c:>16.4f} {carried_caught[m]:>20.4f}")

## Step 6 — Natural confident-wrong on the sibling-absent pool

On the pure-absent pool, *any* named answer is wrong. The certificate's structural abstention keeps its
confident-wrong rate lowest — beating the raw commit, every confidence signal, **and** the query-side verifier
(~3× lower than the verifier on the full pool).

In [ ]:
li_natcw = {
    "certificate":          natural_cw_rate(li_sib, "predict_certificate"),
    "ct_verbalized":        natural_cw_rate(li_sib, "predict_conf_thresh_verbalized"),
    "ct_sc_margin":         natural_cw_rate(li_sib, "predict_conf_thresh_sc_margin"),
    "ct_ptrue":             natural_cw_rate(li_sib, "predict_conf_thresh_ptrue"),
    "ct_negent":            natural_cw_rate(li_sib, "predict_conf_thresh_negent"),
    "queryside_verifier":   natural_cw_rate(li_sib, "predict_queryside_verifier"),
    "queryside_selfverify": natural_cw_rate(li_sib, "predict_queryside_selfverify"),
}
carried_natcw = carried["natural_confident_wrong"]
print(f"{'method':24s} {'conf-wrong (subset)':>20s} {'conf-wrong (full/paper)':>24s}")
print("-" * 70)
print(f"{'raw commit':24s} {factA_sib:>20.4f} {carried_natcw['raw_commit']:>24.4f}")
for m in ["certificate", "ct_verbalized", "queryside_verifier", "queryside_selfverify"]:
    print(f"{m:24s} {li_natcw[m]:>20.4f} {carried_natcw[m]:>24.4f}")

## Step 7 — Doc-clustered paired bootstrap of every caught-gap

Reproduces the STEP-2 headline statistic: for each competitor, the *certificate − competitor* caught-gap with a
document-clustered 95% CI (resampling `BOOTSTRAP_B` document-bootstraps under `EVAL_SEED`).

In [ ]:
fab_li = [r for r in li_sib if r.get("metadata_raw_named")]          # the fabrications
fab_doc_ids = [r["metadata_doc_id"] for r in fab_li]
cert_caught_vec = np.array([1.0 if not is_named(r["predict_certificate"]) else 0.0 for r in fab_li], float)
tau = {s: li_crux["per_method"][f"ct_{s}"]["tau_global"] for s in SIGNALS}

caught_gaps = {}
# dispersion signals: catch == signal value below the certificate-matched threshold
for s in SIGNALS:
    m = f"ct_{s}"
    comp_caught_vec = np.array([1.0 if (float(r.get(f"metadata_conf_{s}", 0.0) or 0.0) < tau[s]) else 0.0
                                for r in fab_li], float)
    caught_gaps[m] = doc_clustered_caught_gap_bootstrap(fab_li, cert_caught_vec, comp_caught_vec,
                                                        fab_doc_ids, B=BOOTSTRAP_B, seed=EVAL_SEED)
# query-side gates: catch == flagged (not named)
for qs in ("queryside_verifier", "queryside_selfverify"):
    comp_caught_vec = np.array([1.0 if not is_named(r[f"predict_{qs}"]) else 0.0 for r in fab_li], float)
    caught_gaps[qs] = doc_clustered_caught_gap_bootstrap(fab_li, cert_caught_vec, comp_caught_vec,
                                                        fab_doc_ids, B=BOOTSTRAP_B, seed=EVAL_SEED)

gaps_excluding_0 = [m for m, g in caught_gaps.items() if g["ci_excludes_0"]]
print(f"certificate - competitor caught-gaps (B={BOOTSTRAP_B}, seed={EVAL_SEED}):\n")
print(f"{'competitor':24s} {'gap':>8s}  {'ci95':>20s}  {'excl 0':>7s}")
print("-" * 64)
for m in ["ct_verbalized", "ct_sc_margin", "ct_ptrue", "ct_negent", "queryside_verifier", "queryside_selfverify"]:
    g = caught_gaps[m]
    print(f"{m:24s} {g['gap']:>8.4f}  [{g['ci95'][0]:>7.3f}, {g['ci95'][1]:>7.3f}]  {str(g['ci_excludes_0']):>7s}")
print(f"\ngaps with CI excluding 0 (subset): {len(gaps_excluding_0)}/6   (full pool: 6/6)")

## Step 8 — Visualization

A bar chart of the caught-fraction leaderboard (subset vs carried full-pool) and the certificate's
confident-wrong advantage. The certificate bar dominates in both views.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# (a) caught-fraction leaderboard: subset vs carried full-pool
order = sorted(METHODS, key=lambda m: -subset_caught[m])
x = np.arange(len(order))
w = 0.4
sub_vals = [subset_caught[m] for m in order]
full_vals = [carried_caught[m] for m in order]
colors = ["#1b7837" if m == "certificate" else "#888888" for m in order]
axes[0].bar(x - w/2, sub_vals, w, label="subset (demo)", color=colors, edgecolor="black")
axes[0].bar(x + w/2, full_vals, w, label="full pool (paper)", color=colors, alpha=0.45, edgecolor="black", hatch="//")
axes[0].set_xticks(x)
axes[0].set_xticklabels([LABELS[m] for m in order], rotation=35, ha="right", fontsize=8)
axes[0].set_ylabel("fraction of fabrications caught")
axes[0].set_title(f"Caught-fraction leaderboard (n_fab={n_fab} subset / 135 full)")
axes[0].set_ylim(0, 1.0)
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

# (b) natural confident-wrong on the sibling-absent pool (lower is better)
cw_methods = ["raw commit", "certificate", "ct_verbalized", "queryside_verifier", "queryside_selfverify"]
cw_sub = [factA_sib, li_natcw["certificate"], li_natcw["ct_verbalized"],
          li_natcw["queryside_verifier"], li_natcw["queryside_selfverify"]]
cw_colors = ["#888888", "#1b7837", "#888888", "#888888", "#888888"]
axes[1].bar(np.arange(len(cw_methods)), cw_sub, color=cw_colors, edgecolor="black")
axes[1].set_xticks(np.arange(len(cw_methods)))
axes[1].set_xticklabels(cw_methods, rotation=35, ha="right", fontsize=8)
axes[1].set_ylabel("confident-wrong rate (lower = better)")
axes[1].set_title("Natural confident-wrong (sibling-absent subset)")
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("HEADLINE (subset): closure certificate catches the LARGEST fraction of fabrications "
      f"({subset_caught['certificate']:.3f}) and keeps confident-wrong LOWEST "
      f"({li_natcw['certificate']:.3f}) — the full pool reproduces certificate caught 0.785 vs verifier 0.274.")